# Day 4: Transformers Library Models

> This notebook documents **Day 4 only**.  
> Later days have separate notebooks.

## Overview

This notebook explores the **lower-level API** of Transformers - the models that wrap PyTorch code for the transformers themselves. This is the step beyond pipelines, where we directly interact with model objects.

## Learning Objectives

- Understand the lower-level Transformers API (AutoTokenizer, AutoModelForCausalLM)
- Learn about quantization (4-bit quantization with BitsAndBytesConfig)
- Explore Transformer model architecture (layers, embeddings, decoder layers)
- Use TextStreamer for streaming outputs
- Understand generation prompts (add_generation_prompt=True)
- Work with multiple models: Llama 3.2, Phi-4, Gemma, Qwen, DeepSeek
- Learn memory management techniques
- Understand model memory footprint

## Resources

- [Models Colab](https://colab.research.google.com/drive/1hhR9Z-yiqjUe7pJjVQw4c74z_V3VchLy?usp=sharing)
- [HuggingFace Transformers Docs](https://huggingface.co/docs/transformers)
- [Llama Model Code](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py)

## Colab Pro-Tip: Misleading CUDA Errors

In the middle of running a Colab, you might get an error like:
```
Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]
```

**This is a super-misleading error message!** Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. `Kernel menu → Disconnect and delete runtime`
2. Reload the colab from fresh and `Edit menu → Clear All Outputs`
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

## Setup

### Install Dependencies

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate

### Imports

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

### HuggingFace Authentication

If you haven't already done so, create a free HuggingFace account at https://huggingface.co and navigate to Settings, then Create a new API token, giving yourself write permissions by clicking on the WRITE tab.

Press the "key" icon on the side panel to the left, and add a new secret: HF_TOKEN = your_token

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Model Selection

Select which model you want to use. Llama models require Meta approval (see access instructions below).

In [ ]:
# Instruct models and 1 reasoning model

# Llama 3.1 is larger and you should already be approved
# see here: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct
# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but you might need to request access again
# see here: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
]

## Accessing Llama from Meta

In order to use Llama models, Meta requires you to sign their terms of service.

Visit their model instructions page in Hugging Face: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

At the top of the page are instructions on how to agree to their terms. If possible, you should use the same email as your huggingface account.

In my experience approval comes in a couple of minutes. Once you've been approved for any 3.1 model, it applies to the whole family of models.

If you have any problems accessing Llama, please see this colab: https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8

## Quantization Configuration

Quantization allows us to load models into memory using less memory. This is essential for running larger models on limited GPU memory (like T4).

In [ ]:
# Quantization Config - this allows us to load the model into memory and use less memory
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

## Loading Tokenizer and Model

### Step 1: Load Tokenizer

The tokenizer converts text to token IDs that the model can process.

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

inputs

### Step 2: Load Model with Quantization

The model is the actual neural network. We load it with quantization to fit in GPU memory.

In [ ]:
# The model
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

## Looking Under the Hood: Transformer Architecture

The next cell prints the HuggingFace model object. This model object is a Neural Network, implemented with PyTorch, using the Transformer architecture invented by Google scientists in 2017.

**Key things to notice:**
- It consists of **layers**
- There's something called **"embedding"** - this takes tokens and turns them into high-dimensional vectors (e.g., 4,096 dimensions)
- There are then multiple sets of **"Decoder layers"** (16 for Llama 3.2, 32 for Llama 3.1). Each Decoder layer contains:
  - (a) **self-attention layers**
  - (b) **multi-layer perceptron (MLP) layers**
  - (c) **batch norm layers**
- There is an **LM Head layer** at the end; this produces the output
- Notice the mention that the model has been **quantized to 4 bits**

**Optional Deep Dive:** If you want to go deeper, check out this tutorial: https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

**Even Deeper:** You can look at the actual HuggingFace code that implements Llama:
- Transformers repo: https://github.com/huggingface/transformers
- Llama model code: https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

In [ ]:
# Execute this cell and look at what gets printed; investigate the layers
model

## Generating Text

Now let's run the model to generate text!

In [ ]:
# Generate output
outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

The output is token IDs. We need to decode them back to text:

In [ ]:
# Decode the token IDs back to text
tokenizer.decode(outputs[0])

## Memory Management

It's important to clean up memory when switching between models. This is especially important in Colab where GPU memory is limited.

In [ ]:
# Clean up memory
# Thank you Kuan L. for helping me get this to properly free up memory!
# If you select "Show Resources" on the top right to see GPU memory, it might not drop down right away
# But it does seem that the memory is available for use by new models in the later code.

del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

## TextStreamer and Generation Prompts

**TextStreamer** allows results to stream back in real-time as they're generated, rather than waiting for the entire generation to complete.

**Generation Prompts** (`add_generation_prompt=True`) ensure that the model generates a response to the question, instead of just predicting how the user prompt continues.

Try experimenting with setting `add_generation_prompt=False` to see what happens!

Read more: https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

## Wrapping Everything in a Function

This function encapsulates the model loading and generation process, making it easy to test multiple models.

In [ ]:
# Wrapping everything in a function - and adding Streaming and generation prompts
def generate(model, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

## Testing Multiple Models

### Phi-4 (Microsoft)

In [ ]:
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
]
generate(PHI, messages)

## Accessing Gemma from Google

A student let me know (thank you, Alex K!) that Google also now requires you to accept their terms in HuggingFace before you use Gemma.

Please visit their model page at this link and confirm you're OK with their terms, so that you're granted access: https://huggingface.co/google/gemma-3-270m-it

### Gemma (Google)

In [ ]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
]
generate(GEMMA, messages, quant=False)

### Qwen (Alibaba Cloud)

In [ ]:
generate(QWEN, messages)

### DeepSeek (Reasoning Model)

In [ ]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)

## Key Learnings

1. **Lower-Level API**: `AutoModelForCausalLM` gives direct access to model objects, beyond the high-level pipeline API
2. **Quantization**: 4-bit quantization allows running larger models on limited GPU memory
3. **Model Architecture**: Transformer models consist of embeddings, decoder layers (attention + MLP), and LM head
4. **TextStreamer**: Enables real-time streaming of generated text
5. **Generation Prompts**: `add_generation_prompt=True` ensures models generate responses, not just continue prompts
6. **Memory Management**: Proper cleanup (`del`, `gc.collect()`, `torch.cuda.empty_cache()`) is essential when switching models
7. **Model Access**: Some models (Llama, Gemma) require approval/terms acceptance
8. **Multiple Models**: Different models have different strengths, sizes, and behaviors

# Day 4: Transformers Library Models

## Overview

This notebook explores the transformers library - the heart of HuggingFace, focusing on models.

## Learning Objectives

- Use `AutoModel` and `AutoModelForCausalLM`
- Understand manual inference flow
- Learn model architecture
- Compare pipelines vs manual inference

## Resources

- [Models Colab](https://colab.research.google.com/drive/1hhR9Z-yiqjUe7pJjVQw4c74z_V3VchLy?usp=sharing)
- [HuggingFace Models Docs](https://huggingface.co/docs/transformers/model_doc/auto)


## Setup


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


## Experiments

### 1. Load Model and Tokenizer


In [ ]:
# Load tokenizer and model
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()  # Set to evaluation mode

print(f"Model loaded: {model_name}")
print(f"Device: {device}")
